In [1]:
import gzip
import random


##### FASTA #####
class Fasta:
    def __init__(self, filename):
        self.filename = filename
        self.gene_length = {}
        self.gene_sequences = {}

    def reading_dnaFasta(self):
        infile = open(self.filename, 'r')
        header = ''
        sequence = ''

        for line in infile:
            line = line.strip()
            if line.startswith('>'):
                if header != '':
                    yield header, sequence
                header = line[1:]
                sequence = ''
            else:
                sequence += line
        if sequence != '':
            yield header, sequence

        infile.close()

    def read_fasta_dict(self):
        for header, seq in self.reading_dnaFasta():
            name = header.strip().split(' ')[0]
            self.gene_length[name] = len(seq)
            self.gene_sequences[name] = seq


##### DNA #####
class DNA:
    def reverse_com_strand(self, sequence):
        comp = {'A':'T','T':'A','G':'C','C':'G'}
        rev = []
        for base in sequence[::-1]:
            if base in comp:
                rev.append(comp[base])
            else:
                rev.append('N')

        return ''.join(rev)

    def nineteen_pos(self, DNA_seq):
        for i in range(len(DNA_seq) - 18):
            kmer = DNA_seq[i:i+19]
            if 'N' not in kmer:
                yield i, kmer


##### KMER #####
class Kmer:
    def __init__(self, fasta):
        self.fasta = fasta
        self.kmer_index = {}
        self.dna = DNA()

    def create_dict_kmer(self):
        for header, sequence in self.fasta.reading_dnaFasta():
            gene_name = header.strip().split(' ')[0]

            for position, kmer in self.dna.nineteen_pos(sequence):
                if kmer not in self.kmer_index:
                    self.kmer_index[kmer] = []
                self.kmer_index[kmer].append((gene_name, position, '+'))
            
            rev = self.dna.reverse_com_strand(sequence)
            for rev_position, kmer in self.dna.nineteen_pos(rev):
                if kmer not in self.kmer_index:
                    self.kmer_index[kmer] = []
                self.kmer_index[kmer].append((gene_name, rev_position, '-'))


##### FASTQ #####
class Fastq:
    def __init__(self, filename):
        self.filename = filename

    def fastqread(self):
        infile = gzip.open(self.filename, 'rt')

        while True:
            infile.readline()
            seq = infile.readline().rstrip()
            infile.readline()
            infile.readline()

            if len(seq) == 0:
                break

            yield seq

        infile.close()


##### MATCHING #####
class Match:
    def __init__(self, fastq_files, resistance_kmer, gene_lengths):
        self.fastq_files = fastq_files
        self.resistance_kmer = resistance_kmer
        self.gene_lengths = gene_lengths

        self.dict_depth = {gene: [0] * length for gene, length in gene_lengths.items()}
        self.unique_match_counts = {gene: 0 for gene in gene_lengths}

    def match_reads_to_genes(self):

        for fastq_file in self.fastq_files:
            fastq = Fastq(fastq_file)

            for read in fastq.fastqread():
                read_len = len(read)

                if read_len < 19:
                    continue

                possible_hit = False
                for pos in range(0, read_len - 18, 20):
                    kmer = read[pos:pos + 19]
                    if 'N' in kmer:
                        continue
                    if kmer in self.resistance_kmer:
                        possible_hit = True
                        break
                if not possible_hit:
                    continue

                reads_found_gene = {}

                for read_pos in range(read_len - 18):
                    kmer_read = read[read_pos:read_pos + 19]
                    if 'N' in kmer_read or kmer_read not in self.resistance_kmer:
                        continue

                    genes_matched = self.resistance_kmer[kmer_read]

                    first_gene_matched = genes_matched[0][0]
                    unique = True
                    for entry in genes_matched:
                        if entry[0] != first_gene_matched:
                            unique = False
                            break

                    for gene, gene_pos, strand in genes_matched:
                        alignment_shift = gene_pos - read_pos
                        id = (gene, alignment_shift, strand)

                        if id not in reads_found_gene:
                            reads_found_gene[id] = [0, 0]

                        reads_found_gene[id][0] += 1

                        if unique:
                            reads_found_gene[id][1] += 1

                if not reads_found_gene:
                    continue

                best_hits = 0
                for info in reads_found_gene.values():
                    if info[0] > best_hits:
                        best_hits = info[0]

                if best_hits < 3:
                    continue

                for (gene, shift, strand), (hit_count, unique_hits) in reads_found_gene.items():
                    if hit_count == best_hits:

                        gene_len = self.gene_lengths[gene]

                        if strand == '+':
                            start = shift
                            end = shift + read_len - 1
                        else:
                            start = gene_len - shift - read_len
                            end = gene_len - 1 - shift

                        if start < 0:
                            start = 0
                        if end > gene_len - 1:
                            end = gene_len - 1

                        for p in range(start, end + 1):
                            self.dict_depth[gene][p] += 1

                        self.unique_match_counts[gene] += unique_hits


##### ANALYSIS #####
class Analysis:
    def __init__(self, dict_depth, gene_length, resistance_kmer, dict_unique_hits):
        self.dict_depth = dict_depth
        self.gene_length = gene_length
        self.resistance_kmer = resistance_kmer
        self.dict_unique_hits = dict_unique_hits

    def calculate_coverage_depth(self, min_cov=95, min_depth=10):
        filtered_genes = {}

        for gene, pos_counts in self.dict_depth.items():
            gene_len = self.gene_length[gene]

            core_start = 18
            core_end = gene_len - 18

            if core_end <= core_start:
                continue

            core = pos_counts[core_start:core_end]
            n = len(core)

            covered = sum(1 for d in core if d > 0)
            coverage = covered / n * 100
            avg_depth = sum(core) / n

            if coverage >= min_cov and avg_depth >= min_depth:
                filtered_genes[gene] = {
                    'coverage': coverage,
                    'depth': avg_depth,
                    'min_depth': min(core)
                }

        return filtered_genes

    def join_genes_by_similarity(self, filtered_genes, threshold=0.5):

        gene_kmers = {gene: set() for gene in self.gene_length}
        for kmer, entries in self.resistance_kmer.items():
            for gene, position, strand in entries:
                gene_kmers[gene].add(kmer)

        gene_list = list(filtered_genes.keys())
        removed = set()

        for i in range(len(gene_list)):
            for j in range(i + 1, len(gene_list)):
                gene_a, gene_b = gene_list[i], gene_list[j]

                if gene_a in removed or gene_b in removed:
                    continue

                kmer_a, kmer_b = gene_kmers[gene_a], gene_kmers[gene_b]
                shared = len(kmer_a & kmer_b)

                if shared == 0:
                    continue

                smaller = kmer_a if len(kmer_a) <= len(kmer_b) else kmer_b
                similarity = shared / len(smaller)

                if similarity >= threshold:
                    stat_a, stat_b = filtered_genes[gene_a], filtered_genes[gene_b]

                    if stat_a['coverage'] != stat_b['coverage']:
                        better, worse = (gene_a, gene_b) if stat_a['coverage'] > stat_b['coverage'] else (gene_b, gene_a)
                    elif stat_a['depth'] != stat_b['depth']:
                        better, worse = (gene_a, gene_b) if stat_a['depth'] > stat_b['depth'] else (gene_b, gene_a)
                    else:
                        better, worse = (gene_a, gene_b) if self.dict_unique_hits[gene_a] >= self.dict_unique_hits[gene_b] else (gene_b, gene_a)

                    removed.add(worse)

        final = {}
        for gene in filtered_genes:
            if gene not in removed:
                final[gene] = filtered_genes[gene]

        return final, removed


# ----------------------------
# MAIN
# ----------------------------
if __name__ == '__main__':

    fasta_file = 'resistance_genes.fsa.txt'
    fastq_files = ['Unknown3_raw_reads_1.txt.gz', 'Unknown3_raw_reads_2.txt.gz']

    fasta = Fasta(fasta_file)
    fasta.read_fasta_dict()

    kmer = Kmer(fasta)
    kmer.create_dict_kmer()

    matcher = Match(fastq_files, kmer.kmer_index, fasta.gene_length)
    matcher.match_reads_to_genes()

    analysis = Analysis(matcher.dict_depth, fasta.gene_length,
                        kmer.kmer_index, matcher.unique_match_counts)

    filtered_genes = analysis.calculate_coverage_depth()
    final_genes, removed = analysis.join_genes_by_similarity(filtered_genes)

    sorted_genes = sorted(final_genes.items(),
        key=lambda x: (-x[1]['coverage'], -x[1]['depth'], -x[1]['min_depth']))

    for gene, stats in sorted_genes:
        print(gene, "coverage:", round(stats['coverage'], 2),
              "depth:", round(stats['depth'], 2),
              "min_depth:", round(stats['min_depth'], 2))

catB4_1_EU935739 coverage: 100.0 depth: 144.62 min_depth: 38
fosA_3_ACWO01000079 coverage: 100.0 depth: 140.1 min_depth: 62
oqxA_1_EU370913 coverage: 100.0 depth: 132.53 min_depth: 94
oqxB_1_EU370913 coverage: 100.0 depth: 124.51 min_depth: 80
blaSHV-28_1_HM751101 coverage: 100.0 depth: 91.93 min_depth: 77
tet(A)_4_AJ517790 coverage: 100.0 depth: 70.49 min_depth: 44
aac(3)-IIa_1_CP023555.1 coverage: 100.0 depth: 56.91 min_depth: 43
aac(6')Ib-cr_1_DQ303918 coverage: 100.0 depth: 55.87 min_depth: 38
strB_1_M96392 coverage: 100.0 depth: 53.89 min_depth: 42
dfrA14_1_DQ388123 coverage: 100.0 depth: 52.41 min_depth: 31
blaCTX-M-15_23_DQ302097 coverage: 100.0 depth: 51.15 min_depth: 28
blaTEM-1B_1_JF910132 coverage: 100.0 depth: 50.46 min_depth: 38
strA_4_AF321551 coverage: 100.0 depth: 50.25 min_depth: 33
blaOXA-1_1_J02967 coverage: 100.0 depth: 49.52 min_depth: 36
sul2_2_GQ421466 coverage: 100.0 depth: 45.31 min_depth: 30
